# Quickstart guide for the tiled GEDI database on MAAP

This quickstart guide is intended for users who are already familiar with DuckDB, SQL, and the principles of hive-partitioned parquet datasets; who are looking to get started with the tiled GEDI dataset. If that is not you, please check out `tiling_demo.ipynb`, which contains a more detailed tutorial.

## Setup

```bash
$ git clone https://github.com/ameliaholcomb/gedi_tiler/
$ conda env create -f gedi_tiler/environment.yml
$ conda activate pyduck
$ pip install -e gedi_tiler
```

It is recommended to set up your MAAP node to have at least 16GB of RAM.

## Basic queries

In [1]:
from gtiler.database import ducky

# Set a temporary directory for DuckDB to fall over to disk
# if it runs out of memory. This must be a local directory, not an S3 path.
# Mounted S3 buckets on an ADE/Hub node are fine.
temp_dir = "/tmp/duckdb"

# Create a connection to DuckDB with default settings and extensions enabled.
con = ducky.init_duckdb(temp_dir=temp_dir)
# Get the path to the global V2 GEDI database on S3.
data_spec = ducky.data_spec("maap-ops-workspace", "shared/ameliah/tiled_gedi")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
# Try a sample query
con.sql(f"""--sql
        SELECT shot_number, rh_98, agbd, geometry
        FROM read_parquet('{data_spec}')
        WHERE l4_quality_flag = 1
        """)

┌────────────────────┬───────┬───────────┬──────────────────────────────────────────────┐
│    shot_number     │ rh_98 │   agbd    │                   geometry                   │
│       uint64       │ float │   float   │                   geometry                   │
├────────────────────┼───────┼───────────┼──────────────────────────────────────────────┤
│  28330000100059911 │  8.01 │   33.6804 │ POINT (9.26473423337465 -0.959024442326173)  │
│  28330100100056225 │ 12.58 │  56.00711 │ POINT (9.257653471478436 -0.97871353128708)  │
│  28330100100056242 │ 32.07 │ 294.40402 │ POINT (9.262735942259512 -0.971549441352168) │
│  28330200100060479 │ 19.03 │ 112.72638 │ POINT (9.263501301962282 -0.979445775067347) │
│  28330200100060480 │ 19.44 │ 126.04507 │ POINT (9.2637989554219 -0.97902396973545)    │
│  28330500100056131 │ 17.68 │  90.98829 │ POINT (9.263181413288434 -0.99965275372537)  │
│  28330500100056706 │ 12.36 │  65.45509 │ POINT (9.435446323998844 -0.755194519619302) │
│  2833050

## Spatial queries

Because spatial features are not yet fully supported by DuckDB-python, some small tricks are required to translate between geopandas and DuckDB and to run spatial queries on the database.

### 0. Setup

In [ ]:
import geopandas as gpd
from shapely.geometry import box
from shapely import Point

# Create an example shape -- or can use any shapefile here.
region_gdf = gpd.GeoDataFrame(
    { 
        "id": [1, 2], 
        "geometric_data": [Point([1,2]), Point([3,4])]
    },  
    geometry=[
        box(-92, 35, -89.5, 36),
        box(-92.9, 35.1, -92.1, 35.3)
    ], 
    crs="EPSG:4326")
region_gdf

,id,geometric_data,geometry
0,1,POINT (1 2),"POLYGON ((-89.5 35, -89.5 36, -92 36, -92 35, ..."
1,2,POINT (3 4),"POLYGON ((-92.1 35.1, -92.1 35.3, -92.9 35.3, ..."


### 1. Convert between DuckDB and Geopandas tables

In [ ]:
# A. Geodataframe to DuckDB
#   - geometry_columns is a list of all the columns in the GeoDataFrame containing a geometry type.
region = ducky.gdf_to_duck(con, region_gdf, geometry_columns=["geometry", "geometric_data"])
con.sql("SELECT * FROM region")

┌───────┬────────────────┬────────────────────────────────────────────────────────────────────────┐
│  id   │ geometric_data │                                geometry                                │
│ int64 │    geometry    │                                geometry                                │
├───────┼────────────────┼────────────────────────────────────────────────────────────────────────┤
│     1 │ POINT (1 2)    │ POLYGON ((-89.5 35, -89.5 36, -92 36, -92 35, -89.5 35))               │
│     2 │ POINT (3 4)    │ POLYGON ((-92.1 35.1, -92.1 35.3, -92.9 35.3, -92.9 35.1, -92.1 35.1)) │
└───────┴────────────────┴────────────────────────────────────────────────────────────────────────┘

In [ ]:
# B. DuckDB table to Geodataframe
#   - geometry_columns is a list of all the columns in the DuckDB table containing a geometry type.
#   - Geopandas requires a single primary geometry column; the primary column will
#       be the first geometry column in the list.
#   - DuckDB geometries do not include a CRS, so you must specify the CRS for geopandas.
new_gdf = ducky.duck_to_gdf(region, geometry_columns=["geometry", "geometric_data"], crs="EPSG:4326")
new_gdf

,id,geometric_data,geometry
0,1,POINT (1 2),"POLYGON ((-89.5 35, -89.5 36, -92 36, -92 35, ..."
1,2,POINT (3 4),"POLYGON ((-92.1 35.1, -92.1 35.3, -92.9 35.3, ..."


### 2. Perform geospatial queries

Out of the box, the DuckDB query planner and spatial extension do not optimize well on a large spatially organized database like the tiled GEDI dataset. In order to help the query planner, add a _fast filter_ to all spatial queries as part of the `WHERE` clause.

Be aware: the fast filter will not help to optimize spatial join operations. This feature is expected to have support in future versions of DuckDB and the tiled GEDI dataset, but it is currently awaiting (planned) work in:
- Geoparquet V2 adoption in pyarrow
- Spatial join optimized query planning in the DuckDB spatial extension, including integration with the ducklake metadata service


In [38]:
# Use the ducky library to create a spatial fast filter clause.
# Note that this function operates on a geopandas shape, not a DuckDB table.
fast_filter = ducky.spatial_filter_clause(region_gdf)

con.sql(f"""--sql
        SELECT 
            COUNT(shot_number) AS n_shots, 
            AVG(agbd), 
            AVG(rh_98)
        FROM 
            read_parquet('{data_spec}') g,
            region r
        WHERE 1=1
            AND l4_quality_flag = 1
            AND {fast_filter} 
            AND ST_Contains(r.geometry, g.geometry)
        GROUP BY r.id
        """)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────────┬────────────────────┐
│ n_shots │     avg(agbd)     │     avg(rh_98)     │
│  int64  │      double       │       double       │
├─────────┼───────────────────┼────────────────────┤
│   44528 │  81.7629646227002 │ 12.430214920309092 │
│ 1143958 │ 62.76870766049441 │  9.341854535063261 │
└─────────┴───────────────────┴────────────────────┘